# LMP spread - per-node LightGBM reference

Global LightGBM with the node as a categorical feature, same inputs the GNN gets. Fold 0 is run in the cells below, folds 1-3 in the loop at the end; every run is logged to the `gridcast-lmp-spread` MLflow experiment.

In [ ]:
import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd

from lmp_model.config import PROCESSED_DATA_DIR
from lmp_model.dataset import (
    FEATURES,
    SYSTEM_FEATURES,
    WEATHER_FEATURES,
    load_cache,
    load_nodes,
    query,
)
from lmp_model.features import fit_scalers, prepare
from lmp_model.splits import split_masks, walk_forward_folds

folds = walk_forward_folds()
nodes = load_nodes()
pnode_ids = nodes.index
data = load_cache()
last_verified = pd.Timestamp(query("SELECT max(time) AS t FROM analytics_lmp.lmp_spread")["t"][0])
masks = split_masks(data["times"], folds[0])
scalers = fit_scalers(data, masks["train"])
blocks = {name: prepare(data, mk, scalers) for name, mk in masks.items()}
{name: int(m.sum()) for name, m in masks.items()}

In [ ]:
y = blocks["test"]["y"]
m = blocks["test"]["loss_mask"]
spike = m & (np.abs(y) >= np.quantile(np.abs(y[m]), 0.99))


def evaluate(pred):
    err = np.abs(pred - y)
    return {"mae": err[m].mean(), "spike_mae": err[spike].mean()}


last_day = data["x"][masks["test"]][..., FEATURES.index("spread_last_verified_day")]
pd.DataFrame({"zero": evaluate(np.zeros_like(y)), "last_day": evaluate(last_day)}).T

In [ ]:
names = FEATURES + ["area_forecast"] + WEATHER_FEATURES + SYSTEM_FEATURES + ["node"]


def flatten(block, step=1):
    """One row per kept (hour, node) cell, every `step`-th hour."""
    t = slice(None, None, step)
    hour, node = np.nonzero(block["loss_mask"][t])
    zone = data["node_weather_zone"][node]
    weather = block["weather"][t][hour, np.maximum(zone, 0)]
    weather[zone < 0] = 0
    X = np.empty((len(hour), len(names)), dtype="float32")
    X[:, :9] = block["x"][t][hour, node]
    X[:, 9] = block["area"][t][hour, data["node_area"][node]]
    X[:, 10:16] = weather
    X[:, 16:22] = block["system"][t][hour]
    X[:, 22] = node
    return X, block["y"][t][hour, node], hour, node


X_train, y_train, _, _ = flatten(blocks["train"], step=3)
X_val, y_val, _, _ = flatten(blocks["val"])
X_test, y_test, hour_test, node_test = flatten(blocks["test"])
X_train.shape, X_val.shape, X_test.shape

In [ ]:
params = {
    "objective": "huber", "alpha": 20.0, "metric": "l1", "learning_rate": 0.1,
    "num_leaves": 63, "num_threads": 4, "verbose": -1,
}
FEATURE_SETS = {"all": names, "core4": FEATURES[:4] + ["node"]}
mlflow.set_experiment("gridcast-lmp-spread")


def train_and_log(i, feature_set="all"):
    fold = folds[i]
    used = FEATURE_SETS[feature_set]
    cols = [names.index(n) for n in used]
    with mlflow.start_run(run_name=f"lgbm_{feature_set}_{pd.Timestamp.now():%Y%m%d}") as run:
        out = PROCESSED_DATA_DIR / "runs" / run.info.run_id
        out.mkdir(parents=True)
        mlflow.set_tags({"fold": i, "feature_set": feature_set, "model": "lgbm_global_node_categorical", "local_artifacts": str(out)})
        mlflow.log_params({
            **params,
            "train_start": fold.train_start, "val_start": fold.val_start,
            "test_start": fold.test_start, "test_end": fold.test_end, "data_end": last_verified,
            "train_hour_step": 3, "n_nodes": len(pnode_ids), "features": ",".join(used),
            "clip_percentiles": "0.1,99.9", "nan_fill": 0, "spike_quantile": 0.99,
            "n_train_rows": len(y_train), "n_val_rows": len(y_val), "n_test_rows": len(y_test),
        })

        evals = {}
        train_set = lgb.Dataset(X_train[:, cols], y_train, feature_name=used, categorical_feature=["node"])
        val_set = lgb.Dataset(X_val[:, cols], y_val, reference=train_set)
        model = lgb.train(
            params, train_set, num_boost_round=1000, valid_sets=[val_set],
            callbacks=[lgb.early_stopping(50), lgb.record_evaluation(evals), lgb.log_evaluation(50)],
        )
        for step, v in enumerate(evals["valid_0"]["l1"]):
            mlflow.log_metric("val_mae_curve", v, step=step)

        pred = np.zeros_like(y)
        pred[hour_test, node_test] = model.predict(X_test[:, cols])
        test = evaluate(pred)
        zero = evaluate(np.zeros_like(y))
        last = evaluate(last_day)
        mlflow.log_metrics({
            "best_iteration": model.best_iteration,
            "val_mae": model.best_score["valid_0"]["l1"],
            "test_mae": test["mae"], "test_spike_mae": test["spike_mae"],
            "zero_mae": zero["mae"], "zero_spike_mae": zero["spike_mae"],
            "last_day_mae": last["mae"], "last_day_spike_mae": last["spike_mae"],
            **{f"loss_mask_coverage_{name}": float(b["loss_mask"].mean()) for name, b in blocks.items()},
            **{f"gain_{n}": g for n, g in zip(used, model.feature_importance("gain"))},
        })

        model.save_model(str(out / "model.txt"))
        np.savez(out / "scalers.npz", **{f"{k}_{n}": a for k, v in scalers.items() for n, a in zip(["lo", "hi", "mean", "std"], v)})
    summary = {"fold": i, "feature_set": feature_set, "best_iteration": model.best_iteration, **test, "zero_mae": zero["mae"], "zero_spike_mae": zero["spike_mae"]}
    return summary, pred

In [ ]:
summary, pred = train_and_log(0)
summary

In [ ]:
summary_core4, pred_core4 = train_and_log(0, "core4")
summary_core4

In [ ]:
zero_err = np.where(m, np.abs(y), 0)
model_err = np.where(m, np.abs(y - pred), 0)
test_times = pd.DatetimeIndex(data["times"], tz="UTC")[masks["test"]].tz_convert("America/New_York")


def gain_by(keys, axis):
    g = pd.DataFrame({"zero": zero_err.sum(axis), "model": model_err.sum(axis), "n": m.sum(axis)}).groupby(keys).sum()
    g["rel_gain"] = 1 - g["model"] / g["zero"]
    g["share_of_total_gain"] = (g["zero"] - g["model"]) / (zero_err.sum() - model_err.sum())
    return g[["n", "rel_gain", "share_of_total_gain"]].round(3)


spike_split = pd.DataFrame(
    {"zero": [zero_err[spike].sum(), zero_err[~spike].sum()], "model": [model_err[spike].sum(), model_err[~spike].sum()],
     "n": [spike.sum(), (m & ~spike).sum()]}, index=["spike", "non_spike"],
)
spike_split["rel_gain"] = 1 - spike_split["model"] / spike_split["zero"]
spike_split["share_of_total_gain"] = (spike_split["zero"] - spike_split["model"]) / (zero_err.sum() - model_err.sum())
print(spike_split[["n", "rel_gain", "share_of_total_gain"]].round(3))
print(gain_by(test_times.hour, 1))
print(gain_by(test_times.month, 1))
print(gain_by(nodes["zone"].fillna("none").to_numpy(), 0))

In [ ]:
results = []
for i in (1, 2, 3):
    masks = split_masks(data["times"], folds[i])
    scalers = fit_scalers(data, masks["train"])
    blocks = {name: prepare(data, mk, scalers) for name, mk in masks.items()}
    X_train, y_train, _, _ = flatten(blocks["train"], step=3)
    X_val, y_val, _, _ = flatten(blocks["val"])
    X_test, y_test, hour_test, node_test = flatten(blocks["test"])
    y = blocks["test"]["y"]
    m = blocks["test"]["loss_mask"]
    spike = m & (np.abs(y) >= np.quantile(np.abs(y[m]), 0.99))
    last_day = data["x"][masks["test"]][..., FEATURES.index("spread_last_verified_day")]
    results.append(train_and_log(i)[0])
pd.DataFrame(results)